In [1]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 18.1 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from dotenv import load_dotenv, find_dotenv
import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [3]:
fw = load_dataset('HuggingFaceFW/fineweb', name="sample-10BT",
    split="train",
    streaming=True,
)

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [4]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [5]:
fw.features

{'text': Value('string'),
 'id': Value('string'),
 'dump': Value('string'),
 'url': Value('string'),
 'date': Value('string'),
 'file_path': Value('string'),
 'language': Value('string'),
 'language_score': Value('float64'),
 'token_count': Value('int64')}

In [6]:
model = SentenceTransformer('all-MiniLM-L6-V2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
model.get_sentence_embedding_dimension()

/tmp/ipykernel_1614/3073143174.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


384

In [8]:
pc = Pinecone(api_key='pcsk_4i1CNG_5RRPZWLANT5epU8gpeAdkX21pTobAXfhZizzgRTjEhS62mw639agd3P4pKawYhs')

In [9]:
index_name = 'text'

In [10]:
if index_name in [index.name for index in pc.list_indexes()]:
  pc.delete_index(index_name)
  print(f'{index_name} successfully deleted')
else:
  print(f'{index_name} not in index list')

text successfully deleted


In [11]:
pc.create_index(
    name = index_name,
    dimension = model.get_sentence_embedding_dimension(),
    metric = 'cosine',
    spec = ServerlessSpec(
        cloud = 'aws',
        region = 'us-east-1'
    )
)

/tmp/ipykernel_1614/1513041920.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension(),


Name:,text
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://text-cpexjor.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [12]:
index = pc.Index(name='text')

In [13]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [14]:
subset_size = 10000 #no of items you want to process

vectors_to_be_upserted = []

for i, item in enumerate(fw):
  if i >= subset_size:
    break
  text = item['text']
  #print(f'Text: {text}')
  unique_id = str(item['id'])
  language = item['language']

  embedding = model.encode(text,show_progress_bar=False).tolist()
  print(f'Embeddings of current text: {embedding}')

  metadata = {'language':language}

  vectors_to_be_upserted.append((unique_id,embedding,metadata))

batch_size = 1000
for i in range(0,len(vectors_to_be_upserted),batch_size):
  batch = vectors_to_be_upserted[i:i+batch_size]
  index.upsert(vectors =batch)


print(f'Subset of data upserted to Pinecone index: {index_name}')

Output hidden; open in https://colab.research.google.com to view.